# D1 — Verifikasi `src/evaluate.py` — Anggota D
**Versi data:** V3.1_A2_PROVISIONAL. **Split:** hanya LATIH dan VALIDASI; UJI tidak dibaca.

**Tujuan:** membuktikan bahwa `prototipe/anggota_c/src/evaluate.py` menghasilkan angka yang sama dengan hasil resmi Anggota B.

**Cara:** skrip B2 dan B3 milik B dijalankan **tanpa diubah** dengan modul ini, lalu setiap angka dibandingkan dengan file di `results/anggota_b/` (toleransi 0,000001, karena B membulatkan 6 desimal).

Waktu jalan sekitar 5 menit di Colab gratis (CPU).

In [ ]:
%pip -q install scikit-learn==1.6.1 pandas numpy

## 1. Ambil repo
Isi `REPO_URL` dengan alamat GitHub repo kelompok, atau kosongkan untuk mengunggah zip repo.

**Penting:** repo harus sudah berisi file dari `PAKET_D1_EVALUATE.zip` (diekstrak di root repo). Zip buatan Windows juga bisa dipakai. Jika file D1 tidak ditemukan, sel ini akan menampilkan file mana yang kurang.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = ""   # contoh: "https://github.com/<akun>/Sistem-Deteksi-Berita-Hoaks-Berbahasa-Indonesia.git"
BRANCH = "main"

work = Path("/content/repo") if Path("/content").exists() else Path("repo")
TARGET = "d1_verifikasi_evaluate.py"

def extract(zpath, dest):
    """Ekstrak zip; pemisah '\\' dari zip buatan Windows diubah menjadi '/'."""
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            if name.endswith("/") or "__MACOSX" in name:
                continue
            out = dest / name
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)

def find_repo():
    hits = [p for p in work.rglob(TARGET) if p.parent.name == "anggota_d" and p.parent.parent.name == "scripts"]
    return hits[0].parents[2] if hits else None

# Bersihkan sisa percobaan sebelumnya yang gagal
if work.exists() and find_repo() is None:
    shutil.rmtree(work)

if not work.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(work)], check=True)
    else:
        from google.colab import files
        up = files.upload()
        work.mkdir(parents=True)
        extract(next(iter(up)), work)

REPO_DIR = find_repo()
if REPO_DIR is None:
    print("Isi teratas zip/repo:", sorted(p.name for p in work.iterdir())[:20])
    for name in ["evaluate.py", "test_evaluate.py", TARGET, "D1_Verifikasi_Evaluate.ipynb", "B2_run_experiments.py"]:
        hits = list(work.rglob(name))
        print(f"  {name:32s}", "ADA: " + str(hits[0].relative_to(work)) if hits else "TIDAK ADA")
    raise FileNotFoundError(
        "scripts/anggota_d/d1_verifikasi_evaluate.py tidak ada. Kemungkinan besar isi PAKET_D1_EVALUATE.zip "
        "belum diekstrak ke root repo sebelum repo di-zip / di-push. Ekstrak paket itu di root repo lalu ulangi.")
assert (REPO_DIR / "prototipe/anggota_c/src/evaluate.py").is_file(), "prototipe/anggota_c/src/evaluate.py belum ada di repo"
assert (REPO_DIR / "notebooks/anggota_b/scripts/B2_run_experiments.py").is_file(), "skrip B tidak ada di repo"
print("Repo:", REPO_DIR)

## 2. Hubungkan Google Drive dan pilih dataset
Gunakan file V3.1 yang sama dengan A dan B. Skrip akan mencetak apakah SHA-256-nya sama dengan file A.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Bukan Google Colab: gunakan lokasi CSV di perangkat ini.")

DATASET_PATH = "/content/drive/MyDrive/Kelompok 5 - PSP/DATASET_V3_1_KANDIDAT_OPERASIONAL.csv"
assert Path(DATASET_PATH).is_file(), f"CSV tidak ditemukan. Ubah DATASET_PATH: {DATASET_PATH}"

## 3. Jalankan B2 dan B3 dengan `evaluate.py`, lalu bandingkan

In [ ]:
OUTDIR = Path("/content/HASIL_D1") if Path("/content").exists() else Path("HASIL_D1")
r = subprocess.run([sys.executable, str(REPO_DIR / "scripts/anggota_d/d1_verifikasi_evaluate.py"),
                    "--dataset", DATASET_PATH, "--outdir", str(OUTDIR)],
                   capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise RuntimeError("Skrip gagal, lihat pesan di atas")

## 4. Ringkasan
Cara membaca:
- `total_beda = 0` dan `sama_dengan_file_a = true` berarti `evaluate.py` identik dengan modul yang dipakai B.
- Jika ada beda, lihat kolomnya di tabel. Beda kecil hanya pada `ece`, `mean_prob_*`, atau `count_*` biasanya berasal dari pelatihan LinearSVC yang sedikit berbeda antarmesin, bukan dari rumus metrik. Beda pada `macro_f1`, `macro_f1_ci_95`, `precision_*`, atau `expected_loss` untuk baris yang sama berarti rumusnya berbeda: laporkan ke D.

In [ ]:
import json, pandas as pd
s = json.loads((OUTDIR / "D1_RINGKASAN.json").read_text())
print(json.dumps({k: v for k, v in s.items() if k != "per_file"}, indent=2))
display(pd.DataFrame(s["per_file"]))
det = pd.read_csv(OUTDIR / "D1_VERIFIKASI_EVALUATE.csv")
display(det[~det["cocok"]])

## 5. Unduh hasil
Model `.joblib` hasil jalan ulang tidak ikut di zip (ukurannya besar dan bukan hasil resmi).

In [ ]:
zpath = OUTDIR.parent / "HASIL_D1_VERIFIKASI.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTDIR.rglob("*"):
        if p.is_file() and p.suffix != ".joblib":
            z.write(p, p.relative_to(OUTDIR.parent))
print("Zip:", zpath)
try:
    from google.colab import files
    files.download(str(zpath))
except ImportError:
    pass